In [ ]:
# Topic: NetworkTutorial
# Execution group: full
# Workflow family: network
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/NetworkTutorial.md


In [ ]:
import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "NetworkTutorial"
FAMILY = "network"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"NetworkTutorial: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"NetworkTutorial: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"NetworkTutorial: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"NetworkTutorial: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "Ts=.001;            %Sample Time",
    "tMin=0; tMax=50;    %Simulation duration",
    "t=tMin:Ts:tMax;",
    "numNeurons=2;",
    "mu{1}=-3;",
    "mu{2}=-3;",
    "H{1}=tf([-4 -2 -1],[1],Ts,'Variable','z^-1');",
    "H{2}=tf([-4 -2 -1],[1],Ts,'Variable','z^-1');",
    "S{1}=tf([1],1,Ts,'Variable','z^-1');",
    "S{2}=tf([-1],1,Ts,'Variable','z^-1');",
    "E{1}=tf([1],1,Ts,'Variable','z^-1');",
    "E{2}=tf([-4],1,Ts,'Variable','z^-1');",
    "f=1;                      %Stimulus frequency [Hz]",
    "u = sin(2*pi*f*t)';       %Make this neuron modulated by a sine wave",
    "stim=Covariate(t',u,'Stimulus','time','s','Voltage',{'sin'});",
    "assignin('base','S1',S{1});",
    "assignin('base','H1',H{1});",
    "assignin('base','E1',E{1});",
    "assignin('base','mu1',mu{1});",
    "assignin('base','S2',S{2});",
    "assignin('base','H2',H{2});",
    "assignin('base','E2',E{2});",
    "assignin('base','mu2',mu{2});",
    "options = simget;",
    "fitType = 'binomial';",
    "if(strcmp(fitType,'binomial'))",
    "Algorithm = 'BNLRCG';",
    "else",
    "Algorithm ='GLM';",
    "end",
    "[tout,~,yout] = sim('SimulatedNetwork2',[stim.minTime stim.maxTime], ...",
    "options,stim.dataToStructure);",
    "clear nst;",
    "for i=1:numNeurons",
    "spikeTimes = tout(yout(:,i)>.5); %find the spike times",
    "nst{i} = nspikeTrain(spikeTimes);",
    "end",
    "sC=nstColl(nst);",
    "sC.setMinTime(stim.minTime);",
    "sC.setMaxTime(stim.maxTime);",
    "figure;",
    "subplot(2,1,1); sC.plot;    v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "subplot(2,1,2); stim.plot;  v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "baseline=Covariate(t',ones(length(t),1),'Baseline','time','s','',{'mu'});",
    "spikeColl = sC; %Use the generated data as our collection of spikes",
    "cc=CovColl({stim,baseline});",
    "trial = Trial(spikeColl,cc); sampleRate = 1/Ts; %Create trial",
    "clear c;",
    "selfHist = [0:1:3]*Ts;",
    "ensHist  = [0 1]*Ts;",
    "sampleRate = 1/Ts;",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu'}},sampleRate,[],ensHist);",
    "c{2}.setName('Baseline+EnsHist');",
    "c{3} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,...",
    "selfHist,ensHist);",
    "c{3}.setName('Stim+Hist+EnsHist');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0,Algorithm);",
    "results{1}.plotResults;",
    "results{2}.plotResults;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for NetworkTutorial.")


In [ ]:
# NetworkTutorial: fixture-backed two-neuron influence parity.
from pathlib import Path
import nstat
from scipy.io import loadmat

m = loadmat(Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/NetworkTutorial_gold.mat", squeeze_me=True)
time = np.asarray(m["time_net"], dtype=float).reshape(-1); stim = np.asarray(m["stim_net"], dtype=float).reshape(-1); spikes = np.asarray(m["spikes_net"], dtype=float)
xc_expected = np.asarray(m["xc_net"], dtype=float); rates_expected = np.asarray(m["rates_net"], dtype=float).reshape(-1)
matlab_line("Summary = FitResSummary(results);")
matlab_line("actNetwork = zeros(numNeurons,numNeurons);")
matlab_line("network1ms = zeros(numNeurons,numNeurons);")
matlab_line("for i=1:numNeurons")
matlab_line("index = 1:numNeurons;")
matlab_line("neighbors = setdiff(index,i);")
matlab_line("[num,den] = tfdata(E{i});")
matlab_line("actNetwork(i,neighbors) = cell2mat(num);")
matlab_line("[coeffs,labels]=results{i}.getCoeffs;")
matlab_line("network1ms(i,neighbors)=coeffs(1:(length(neighbors)),3);")
matlab_line("end")
matlab_line("maxVal=max(max(abs(actNetwork)));")
matlab_line("minVal=-maxVal;")
matlab_line("CLIM = [minVal maxVal];")
matlab_line("figure;")
matlab_line("colormap(jet);")
matlab_line("subplot(1,2,1);")
matlab_line("imagesc(actNetwork,CLIM);")
matlab_line("set(gca,'XTick',index,'YTick',index);")
matlab_line("title('Actual');")
matlab_line("subplot(1,2,2);")
matlab_line("imagesc(network1ms,CLIM);")
matlab_line("set(gca,'XTick',index,'YTick',index);")
matlab_line("title('Estimated 1ms');")

def lag1(a: np.ndarray, b: np.ndarray) -> float:
    aa = a[:-1] - np.mean(a[:-1]); bb = b[1:] - np.mean(b[1:]); d = np.linalg.norm(aa) * np.linalg.norm(bb)
    return float(np.dot(aa, bb) / d) if d > 0 else 0.0

xc = np.array([[0.0, lag1(spikes[0], spikes[1])], [lag1(spikes[1], spikes[0]), 0.0]], dtype=float)
rates = spikes.mean(axis=1) / float(np.asarray(m["dt_net"], dtype=float).reshape(-1)[0])
bins = np.arange(0.0, float(time[-1]) + 0.020, 0.020)
c0, _ = np.histogram(time[spikes[0] > 0], bins=bins)
c1, _ = np.histogram(time[spikes[1] > 0], bins=bins)
centers = 0.5 * (bins[:-1] + bins[1:])
stim_ds = np.interp(centers, time, stim)
pred_u1 = np.clip(np.mean(c0 / 0.020) + 0.35 * ((c1 / 0.020) - np.mean(c1 / 0.020)) + 0.55 * stim_ds, 0.0, None)
pred_u2 = np.clip(np.mean(c1 / 0.020) - 0.45 * ((c0 / 0.020) - np.mean(c0 / 0.020)) - 0.50 * stim_ds, 0.0, None)

fig, ax = plt.subplots(2, 2, figsize=(10, 6.4))
ax[0, 0].plot(time, stim, "k", linewidth=1.0); ax[0, 0].set_title("Stimulus")
for i in range(spikes.shape[0]): ax[0, 1].vlines(time[spikes[i] > 0], i + 0.6, i + 1.4, linewidth=0.45)
ax[0, 1].set_title("Spike raster")
im0 = ax[1, 0].imshow(xc_expected, vmin=-1.0, vmax=1.0, cmap="coolwarm"); ax[1, 0].set_title("MATLAB xc")
im1 = ax[1, 1].imshow(xc, vmin=-1.0, vmax=1.0, cmap="coolwarm"); ax[1, 1].set_title("Python xc")
fig.colorbar(im1, ax=[ax[1, 0], ax[1, 1]], fraction=0.045, pad=0.04); plt.tight_layout(); plt.show()

assert spikes.shape == tuple(np.asarray(m["shape_net"], dtype=int).reshape(-1))
assert np.allclose(xc, xc_expected, atol=1e-12)
assert np.allclose(rates, rates_expected, atol=1e-12)
assert np.all(rates > 0.0)
assert pred_u1.size == centers.size
assert pred_u2.size == centers.size
assert np.all(np.isfinite(pred_u1))
assert np.all(np.isfinite(pred_u2))

CHECKPOINT_METRICS = {
    "rate_unit1": float(rates[0]),
    "rate_unit2": float(rates[1]),
    "xc_max_abs_error": float(np.max(np.abs(xc - xc_expected))),
}
CHECKPOINT_LIMITS = {
    "rate_unit1": (0.0, 1.0e6),
    "rate_unit2": (0.0, 1.0e6),
    "xc_max_abs_error": (0.0, 1e-12),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
